# Model-CatBoost


In this section, I plan to build two CatBoost models for the two age group. The reason behind my choice is that:

+ we have a lot of categorical variables, and CatBoost is famous for its ability to deal with categorical variables
+ Tree model usually has large interpretability even it is ensembling model.

In [1]:
# import required libraries
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

In [2]:
pd.set_option('display.max_columns', None)
pd.options.display.max_colwidth = 500
pd.options.display.max_rows = 100

In [3]:
# import the clean and preprocess functions

from preprocess import preprocess_data
# import constants
from features import numeric_cols, binary_cols, multi_category_cols, parent_child_groups, categorical_cols, identifier_cols

In [4]:
# define some useful functions for evaluation from 04_prediction_lr.ipynb

# define the metrics consistent with paper and some common metrics for imbalanced classification
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    confusion_matrix
)


def pecarn_metrics(y_true, y_prob, threshold=0.02):
    """
    threshold = risk cutoff defining 'predictor present'
    """

    y_pred = (y_prob >= threshold).astype(int)

    TP = np.sum((y_true==1) & (y_pred==1))
    FN = np.sum((y_true==1) & (y_pred==0))
    TN = np.sum((y_true==0) & (y_pred==0))
    FP = np.sum((y_true==0) & (y_pred==1))

    sensitivity = TP / (TP + FN) if (TP+FN)>0 else np.nan
    npv = TN / (TN + FN) if (TN+FN)>0 else np.nan
    roc_auc = roc_auc_score(y_true, y_prob)
    pr_auc_score =average_precision_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    specificity = TN / (TN + FP) if (TN+FP)>0 else np.nan

    return {
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "NPV": npv,
        "ROC AUC": roc_auc,
        "PR AUC": pr_auc_score,
        "Precision":  precision,
        "TP": TP,
        "FN": FN,
        "TN": TN,
        "FP": FP
    }


def scan_thresholds(y_true, y_prob,
                    thresholds=np.linspace(0.001, 0.2, 200)):
    rows = []
    y_true = np.asarray(y_true).astype(int)

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        sens = tp / (tp + fn) if (tp + fn) else np.nan
        spec = tn / (tn + fp) if (tn + fp) else np.nan
        rows.append({"thr": t, "sensitivity": sens, "specificity": spec,
                     "youdenJ": sens + spec - 1})
    return pd.DataFrame(rows)


# combine the two logics together
def univariate_feature_screening(features,
                                 target_col,
                                 data,
                                 near_constant_threshold = 0.99,
                                 risk_nodiff_threshold = 0.01):
    omit_cols = set()
    
    for col in features:
        # A: near-constant
        mode_rt = data[col].value_counts(normalize=True, dropna=False).iloc[0]
        if mode_rt > near_constant_threshold:
            omit_cols.add(col)
            continue
        # B: little risk difference
        grp = data.groupby(col)[target_col].mean()
        if len(grp) >= 2:
            risk_diff = grp.max()-grp.min()
            if risk_diff < risk_nodiff_threshold:
                omit_cols.add(col)

    return omit_cols
    

## Data Preparation

After cleaning:

+ numerical: impute GCS component and leave remainging `NaN`
+ categorical: simply leave `NaN`

In [5]:
%run clean.py

In [6]:
tbi_cleaned_new = pd.read_csv('../data/TBI_cleaned.csv')

In [7]:
train_cat_data, val_cat_data, test_cat_data = preprocess_data(
    df=tbi_cleaned_new,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13= False, # include GCS 3-13
    if_fill_missing_gcs=True,
    gcs_fill_strategy="leave", # leave missing 
    if_one_hot_encode=False,
    drop_first_cat_in_ohe=False,
    if_drop_na_rows=False,
    if_handle_parent_child_missing=False,
    if_handle_missing_for_categories=False,
    missing_category_label="missing",
    if_add_flag_missing_cols=False,
    parent_missing_strategy="leave",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="leave",
    multi_missing_strategy="missing_category"
)

In [8]:
train_cat_data.shape, val_cat_data.shape, test_cat_data.shape

((30365, 126), (4338, 126), (8676, 126))

In [9]:
train_cat_data.isnull().sum().sort_values(ascending=False).head(20)

Ethnicity       11201
Dizzy           11119
ActNorm          2322
Race             2213
LocLen           1795
Observed         1694
Amnesia_verb     1602
LOCSeparate      1344
Drugs            1262
HAStart           965
HASeverity        717
VomitLast         680
Seiz              644
CTSed             637
HemaSize          483
NeuroD            460
HA_verb           454
SFxBas            321
Vomit             305
VomitStart        272
dtype: int64

Now, we choose to keep `NaN` values in our dataset

In [10]:
# we need to check our column types
train_cat_data.dtypes.value_counts()

float64    126
Name: count, dtype: int64

In [11]:
# some columns are unrelated to the prediction task or post-CT variabels, we can drop them directly
target_col = 'PosIntFinal'
downstream_var = ([

    # --- ED management / utilization outcomes ---
    "Observed",          # observed in ED
    "EDDisposition",     # discharge / admit / OR etc.
    "CTDone",            # any head CT performed
    "EDCT",              # CT performed in ED
    'CTForm1',         

    # --- CT results ---
    "PosCT",             # traumatic finding on CT

    # Individual CT findings (radiology outcomes)
    "Finding1",   # cerebellar hemorrhage
    "Finding2",   # cerebral contusion
    "Finding3",   # cerebral edema
    "Finding4",   # intracerebral hemorrhage
    "Finding5",   # skull diastasis
    "Finding6",   # epidural hematoma
    "Finding7",   # extra-axial hematoma
    "Finding8",   # intraventricular hemorrhage
    "Finding9",   # midline shift
    "Finding10",  # pneumocephalus
    "Finding11",  # skull fracture
    "Finding12",  # subarachnoid hemorrhage
    "Finding13",  # subdural hematoma
    "Finding14",  # traumatic infarction
    "Finding20",  # diffuse injury (PI added)
    "Finding21",  # herniation
    "Finding22",  # shear injury
    "Finding23",  # sigmoid sinus injury

    # --- Clinical outcomes ---
    "DeathTBI",        # death due to TBI
    "HospHead",        # hospitalized ≥2 nights
    "HospHeadPosCT",   # hospitalization with positive CT
    "Intub24Head",     # intubated >24h
    "Neurosurgery",    # neurosurgical intervention

    # flag for target inconsistency
    "PosIntFinal_invalid"
] + [col for col in train_cat_data.columns if col.startswith("Ind")] # exclude CT indication
+ [col for col in train_cat_data.columns if col.startswith("CTSed")] 
)

In [12]:
drop_cols = downstream_var + identifier_cols + ['AgeInMonth'] # we keep age in month for better granularity
can_cols = [col for col in train_cat_data.columns if col not in drop_cols + [target_col]]

In [13]:
len(can_cols), can_cols[:10]

(70,
 ['InjuryMech',
  'High_impact_InjSev',
  'Amnesia_verb',
  'LOCSeparate',
  'LocLen',
  'Seiz',
  'SeizOccur',
  'SeizLen',
  'ActNorm',
  'HA_verb'])

In [14]:
train_cat_data.drop(columns=drop_cols, inplace=True)
val_cat_data.drop(columns=drop_cols, inplace=True)
test_cat_data.drop(columns=drop_cols, inplace=True)

In [15]:
# define the numeric and categorical columns for modeling
model_numeric_cols = [col for col in can_cols if col in numeric_cols]
model_categorical_cols = [col for col in can_cols if col in categorical_cols]

# convert column types
train_cat_data[model_numeric_cols] = train_cat_data[model_numeric_cols].apply(pd.to_numeric, errors='coerce')
# convert category columns to string
train_cat_data[model_categorical_cols] = (
    train_cat_data[model_categorical_cols]
        .apply(lambda col: col.astype("string").astype("category"))
)
# fill NaN with `missing`
train_cat_data[model_categorical_cols] = (
    train_cat_data[model_categorical_cols]
        .apply(lambda col: col.cat.add_categories("missing").fillna("missing"))
)

In [16]:
# val 
val_cat_data[model_numeric_cols] = val_cat_data[model_numeric_cols].apply(pd.to_numeric, errors='coerce')
val_cat_data[model_categorical_cols] = val_cat_data[model_categorical_cols].apply(lambda col: col.astype("string").astype("category"))

val_cat_data[model_categorical_cols] = (
    val_cat_data[model_categorical_cols]
        .apply(lambda col: col.cat.add_categories("missing").fillna("missing"))
)  
# test
test_cat_data[model_numeric_cols] = test_cat_data[model_numeric_cols].apply(pd.to_numeric, errors='coerce')
test_cat_data[model_categorical_cols] = test_cat_data[model_categorical_cols].apply(lambda col: col.astype("string").astype("category"))
test_cat_data[model_categorical_cols] = (
    test_cat_data[model_categorical_cols]
        .apply(lambda col: col.cat.add_categories("missing").fillna("missing"))
)  

In [17]:
# sanity check
train_cat_data.dtypes.value_counts()

category    29
category    15
category     6
float64      6
category     4
category     4
category     2
category     2
category     1
category     1
category     1
Name: count, dtype: int64

In [18]:
# split age-group 
train_cat_data_age2plus = train_cat_data[train_cat_data['AgeTwoPlus'] == '2.0']
train_cat_data_age2minus = train_cat_data[train_cat_data['AgeTwoPlus'] == '1.0']

val_cat_data_age2plus = val_cat_data[val_cat_data['AgeTwoPlus'] == '2.0']
val_cat_data_age2minus = val_cat_data[val_cat_data['AgeTwoPlus'] == '1.0']

test_cat_data_age2plus = test_cat_data[test_cat_data['AgeTwoPlus'] == '2.0']
test_cat_data_age2minus = test_cat_data[test_cat_data['AgeTwoPlus'] == '1.0']

In [19]:
# drop age group indicator
train_cat_data_age2plus.drop(columns=['AgeTwoPlus'], inplace=True)
train_cat_data_age2minus.drop(columns=['AgeTwoPlus'], inplace=True)
val_cat_data_age2plus.drop(columns=['AgeTwoPlus'], inplace=True)
val_cat_data_age2minus.drop(columns=['AgeTwoPlus'], inplace=True)
test_cat_data_age2plus.drop(columns=['AgeTwoPlus'], inplace=True)
test_cat_data_age2minus.drop(columns=['AgeTwoPlus'], inplace=True)
# also remove the age group indicator from the candidate columns
can_cols.remove('AgeTwoPlus')

In [20]:
train_cat_data_age2plus.shape, train_cat_data_age2minus.shape, val_cat_data_age2plus.shape, val_cat_data_age2minus.shape, test_cat_data_age2plus.shape, test_cat_data_age2minus.shape

((22734, 70), (7631, 70), (3248, 70), (1090, 70), (6496, 70), (2180, 70))

Overall, we have 69 feature columns and 1 target columns.

## Age < 2

### Feature Selection

#### Univariate Sreen

The same logic as we have done in logistic regression.

In [21]:
omit_cols_age2minus = univariate_feature_screening(can_cols, target_col, train_cat_data_age2minus)
omit_cols_age2minus

{'AgeinYears',
 'ClavFace',
 'ClavFro',
 'Ethnicity',
 'Gender',
 'HASeverity',
 'HAStart',
 'NeuroDCranial',
 'NeuroDMotor',
 'NeuroDOth',
 'NeuroDReflex',
 'NeuroDSensory',
 'Paralyzed',
 'SFxBasHem',
 'SFxBasOto',
 'SFxBasPer',
 'SFxBasRet',
 'SFxBasRhi',
 'SFxPalpDepress'}

In [22]:
can_cols_age2minus = list(set(can_cols) - set(omit_cols_age2minus))
len(can_cols_age2minus)

50

Overall, we have 50 candidate features for modeling age<2.

In [23]:
# define the cat feature dict for catboost
cat_features_age2minus = list(set(can_cols_age2minus).intersection(model_categorical_cols))
num_features_age2minus = list(set(can_cols_age2minus).intersection(model_numeric_cols))

In [24]:
print(cat_features_age2minus)

['OSIOth', 'OSIExtremity', 'OSIAbdomen', 'Hema', 'AMSAgitated', 'HemaLoc', 'HemaSize', 'Drugs', 'Dizzy', 'ClavTem', 'LOCSeparate', 'Sedated', 'LocLen', 'VomitStart', 'AMS', 'AMSRepeat', 'OSIPelvis', 'NeuroD', 'Intubated', 'OSICspine', 'AMSOth', 'AMSSleep', 'OSIFlank', 'SeizOccur', 'SFxBas', 'High_impact_InjSev', 'GCSGroup', 'ClavNeck', 'AMSSlow', 'InjuryMech', 'SFxPalp', 'Amnesia_verb', 'Clav', 'Vomit', 'Race', 'FontBulg', 'VomitLast', 'HA_verb', 'ClavPar', 'OSI', 'OSICut', 'ClavOcc', 'Seiz', 'SeizLen', 'VomitNbr', 'ActNorm']


In [25]:
print(num_features_age2minus)

['GCSTotal', 'GCSEye', 'GCSMotor', 'GCSVerbal']


In [26]:
cat_features_age2minus.sort()
num_features_age2minus.sort()
can_cols_age2minus.sort()

### Modeling

#### Baseline

In [27]:
cat_model_age2minus = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    verbose=200
)

cat_model_age2minus.fit(
    train_cat_data_age2minus[can_cols_age2minus],
    train_cat_data_age2minus[target_col],
    eval_set=(val_cat_data_age2minus[can_cols_age2minus], val_cat_data_age2minus[target_col]),
    cat_features= cat_features_age2minus
)

0:	learn: 0.8612624	test: 0.8466314	best: 0.8466314 (0)	total: 69.6ms	remaining: 2m 19s
200:	learn: 0.9810182	test: 0.9389249	best: 0.9415995 (103)	total: 3.37s	remaining: 30.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9415995431
bestIteration = 103

Shrink model to first 104 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, early_stopping_rounds=100, eval_metric='PRAUC', iterations=2000, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200)

In [28]:
val_cat_data_age2minus['pred_prob'] = cat_model_age2minus.predict_proba(val_cat_data_age2minus[can_cols_age2minus])[:, 1]
pecarn_metrics(val_cat_data_age2minus[target_col], val_cat_data_age2minus['pred_prob'], threshold=0.5)

{'Sensitivity': np.float64(0.8666666666666667),
 'Specificity': np.float64(0.9153488372093024),
 'NPV': np.float64(0.9979716024340771),
 'ROC AUC': 0.9467906976744185,
 'PR AUC': 0.4225410962583196,
 'Precision': 0.125,
 'TP': np.int64(13),
 'FN': np.int64(2),
 'TN': np.int64(984),
 'FP': np.int64(91)}

It looks like the baseline model has achieved the best in our validation set.

Let me do a small parameter tuning for generalization

#### Hyperparamter Tuning


For complex model, I prefer **random search**

In [29]:
param_space = {
    "depth": [4,5,6,7,8],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "l2_leaf_reg": [1,3,5,7,10]
}

import random
cat_age2minus_results = []

for i in range(20):   # only 20 trials needed
    params = {k: random.choice(v) for k,v in param_space.items()}

    model = CatBoostClassifier(
        iterations=2000,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        early_stopping_rounds=100,
        random_seed=42,
        verbose=False,
        **params
    )

    model.fit(
        train_cat_data_age2minus[can_cols_age2minus],
        train_cat_data_age2minus[target_col],
        eval_set=(val_cat_data_age2minus[can_cols_age2minus], val_cat_data_age2minus[target_col]),
        cat_features= cat_features_age2minus
    )

    prob = model.predict_proba(val_cat_data_age2minus[can_cols_age2minus])[:,1]

    m = pecarn_metrics(val_cat_data_age2minus[target_col], prob, threshold=0.5)

    cat_age2minus_results.append({**params, **m})

In [30]:
pd.DataFrame(cat_age2minus_results)\
    .sort_values(["Specificity"], ascending=False)\
    .head()

,depth,learning_rate,l2_leaf_reg,Sensitivity,Specificity,NPV,ROC AUC,PR AUC,Precision,TP,FN,TN,FP
7,8,0.05,1,0.733333,0.933023,0.996028,0.945581,0.431553,0.132530,11,4,1003,72
3,8,0.02,1,0.666667,0.932093,0.995035,0.948434,0.435215,0.120482,10,5,1002,73
9,4,0.05,10,0.800000,0.920930,0.996979,0.950729,0.423073,0.123711,12,3,990,85
13,4,0.05,10,0.800000,0.920930,0.996979,0.950729,0.423073,0.123711,12,3,990,85
11,8,0.02,10,0.800000,0.918140,0.996970,0.947814,0.389173,0.120000,12,3,987,88


Our best parameter combination is 
+ "depth" = 8
+ "learning_rate" = 0.05
+ "l2_leaf_reg" = 3


#### Best

In [31]:
best_cat_model_age2minus = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    random_seed=42,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    verbose=200
)

best_cat_model_age2minus.fit(
    train_cat_data_age2minus[can_cols_age2minus],
    train_cat_data_age2minus[target_col],
    eval_set=(val_cat_data_age2minus[can_cols_age2minus], val_cat_data_age2minus[target_col]),
    cat_features= cat_features_age2minus
)

0:	learn: 0.8612624	test: 0.8466314	best: 0.8466314 (0)	total: 14.5ms	remaining: 28.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9349870206
bestIteration = 34

Shrink model to first 35 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, early_stopping_rounds=100, eval_metric='PRAUC', iterations=2000, l2_leaf_reg=3, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=200)

In [32]:
best_iteration = best_cat_model_age2minus.get_best_iteration()
print("Best iteration:", best_iteration)

Best iteration: 34


In [33]:
best_cat_model_age2minus = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=best_iteration,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    random_seed=42,
    auto_class_weights="Balanced",
    # early_stopping_rounds=100,
    verbose=200
)

best_cat_model_age2minus.fit(
    train_cat_data_age2minus[can_cols_age2minus],
    train_cat_data_age2minus[target_col],
    eval_set=(val_cat_data_age2minus[can_cols_age2minus], val_cat_data_age2minus[target_col]),
    cat_features= cat_features_age2minus
)

0:	learn: 0.8705660	test: 0.8760073	best: 0.8760073 (0)	total: 9.21ms	remaining: 304ms
33:	learn: 0.9638024	test: 0.9328354	best: 0.9331656 (29)	total: 363ms	remaining: 0us

bestTest = 0.9331656182
bestIteration = 29

Shrink model to first 30 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, eval_metric='PRAUC', iterations=34, l2_leaf_reg=3, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=200)

In [34]:
# find the best threshold on val set
val_cat_data_age2minus['pred_prob'] = best_cat_model_age2minus.predict_proba(val_cat_data_age2minus[can_cols_age2minus])[:, 1]
threshold_results = scan_thresholds(val_cat_data_age2minus[target_col], val_cat_data_age2minus['pred_prob'], thresholds=np.linspace(0.001, 0.2, 200))
threshold_results.sort_values(['youdenJ'], ascending=False).head(10)

,thr,sensitivity,specificity,youdenJ
189,0.190,1.0,0.665116,0.665116
188,0.189,1.0,0.665116,0.665116
186,0.187,1.0,0.664186,0.664186
185,0.186,1.0,0.664186,0.664186
187,0.188,1.0,0.664186,0.664186
184,0.185,1.0,0.653953,0.653953
182,0.183,1.0,0.653023,0.653023
180,0.181,1.0,0.653023,0.653023
181,0.182,1.0,0.653023,0.653023
183,0.184,1.0,0.653023,0.653023


The threshold of **0.2** is a reasonable choice

In [35]:
# val
val_cat_data_age2minus['pred_prob'] = best_cat_model_age2minus.predict_proba(val_cat_data_age2minus[can_cols_age2minus])[:, 1]
pecarn_metrics(val_cat_data_age2minus[target_col], val_cat_data_age2minus['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.9333333333333333),
 'Specificity': np.float64(0.6818604651162791),
 'NPV': np.float64(0.9986376021798365),
 'ROC AUC': 0.944186046511628,
 'PR AUC': 0.3049160342023194,
 'Precision': 0.03932584269662921,
 'TP': np.int64(14),
 'FN': np.int64(1),
 'TN': np.int64(733),
 'FP': np.int64(342)}

In [36]:
# test 
test_cat_data_age2minus['pred_prob'] = best_cat_model_age2minus.predict_proba(test_cat_data_age2minus[can_cols_age2minus])[:, 1]
pecarn_metrics(test_cat_data_age2minus[target_col], test_cat_data_age2minus['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.8666666666666667),
 'Specificity': np.float64(0.6627906976744186),
 'NPV': np.float64(0.9972008397480756),
 'ROC AUC': 0.8427131782945736,
 'PR AUC': 0.35803531640570474,
 'Precision': 0.03462050599201065,
 'TP': np.int64(26),
 'FN': np.int64(4),
 'TN': np.int64(1425),
 'FP': np.int64(725)}

In [37]:
# train
train_cat_data_age2minus['pred_prob'] = best_cat_model_age2minus.predict_proba(train_cat_data_age2minus[can_cols_age2minus])[:, 1]
pecarn_metrics(train_cat_data_age2minus[target_col], train_cat_data_age2minus['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.6513359032300944),
 'NPV': np.float64(1.0),
 'ROC AUC': 0.9586065694832144,
 'PR AUC': 0.5100322768937107,
 'Precision': 0.039545953863053825,
 'TP': np.int64(108),
 'FN': np.int64(0),
 'TN': np.int64(4900),
 'FP': np.int64(2623)}

**Remark:**

It looks like our model is too good to be ture. One reason behind that is the relatively small sample size in the age < 2 group, especically for ensembling model.

### Interpretability

In [38]:
# get feature importance
feature_importance = best_cat_model_age2minus.get_feature_importance(prettified=True)
feature_importance.head(15)

,Feature Id,Importances
0,HemaLoc,27.500456
1,LOCSeparate,21.370100
2,AMSRepeat,12.351250
3,ActNorm,8.463450
4,AMSOth,6.481482
5,SFxPalp,5.329597
6,GCSVerbal,3.519380
7,ClavOcc,2.694870
8,Seiz,2.479399
9,AMS,2.393588


**Remark:**
All of the six predictors in the CDR model have been included in the catboost model and ranks very high.



## Age >= 2

### Feature Selection

#### Univariate Screen

In [39]:
omit_cols_age2plus = univariate_feature_screening(can_cols, target_col, train_cat_data_age2plus)
omit_cols_age2plus

{'Ethnicity', 'FontBulg', 'SFxPalpDepress'}

In [40]:
can_cols_age2plus = list(set(can_cols) - set(omit_cols_age2plus))
len(can_cols_age2plus)

66

We have 66 candidate features. The similar circumstance happens when I am doing the logistic regression for age over 2 group. I didn't omit as many features as age under 2. 

In [41]:
# define the cat feature dict for catboost
cat_features_age2plus = list(set(can_cols_age2plus).intersection(model_categorical_cols))
num_features_age2plus = list(set(can_cols_age2plus).intersection(model_numeric_cols))

In [42]:
print(cat_features_age2plus)

['OSIOth', 'OSIExtremity', 'NeuroDSensory', 'Hema', 'OSIAbdomen', 'ClavFace', 'AMSAgitated', 'HemaLoc', 'HemaSize', 'Drugs', 'Dizzy', 'ClavTem', 'NeuroDOth', 'NeuroDCranial', 'LOCSeparate', 'Paralyzed', 'Sedated', 'SFxBasRhi', 'LocLen', 'VomitStart', 'AMS', 'AMSRepeat', 'ClavFro', 'OSIPelvis', 'NeuroD', 'Intubated', 'SFxBasPer', 'OSICspine', 'AMSOth', 'AMSSleep', 'OSIFlank', 'SeizOccur', 'SFxBas', 'HAStart', 'High_impact_InjSev', 'SFxBasOto', 'GCSGroup', 'ClavNeck', 'AMSSlow', 'InjuryMech', 'SFxPalp', 'Amnesia_verb', 'Clav', 'Vomit', 'Race', 'SFxBasRet', 'VomitLast', 'SFxBasHem', 'HA_verb', 'ClavPar', 'OSI', 'OSICut', 'ClavOcc', 'NeuroDMotor', 'Seiz', 'SeizLen', 'VomitNbr', 'Gender', 'ActNorm', 'HASeverity', 'NeuroDReflex']


In [43]:
print(num_features_age2plus)

['GCSEye', 'GCSMotor', 'GCSVerbal', 'GCSTotal', 'AgeinYears']


### Modeling

#### Baseline

In [44]:
cat_model_age2plus = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    verbose=200
)

cat_model_age2plus.fit(
    train_cat_data_age2plus[can_cols_age2plus],
    train_cat_data_age2plus[target_col],
    eval_set=(val_cat_data_age2plus[can_cols_age2plus], val_cat_data_age2plus[target_col]),
    cat_features= cat_features_age2plus
)

0:	learn: 0.9202193	test: 0.9391129	best: 0.9391129 (0)	total: 73.6ms	remaining: 2m 27s
200:	learn: 0.9679127	test: 0.9648981	best: 0.9653795 (163)	total: 9.3s	remaining: 1m 23s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9653794727
bestIteration = 163

Shrink model to first 164 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=6, early_stopping_rounds=100, eval_metric='PRAUC', iterations=2000, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200)

In [45]:
val_cat_data_age2plus['pred_prob'] = cat_model_age2plus.predict_proba(val_cat_data_age2plus[can_cols_age2plus])[:, 1]
pecarn_metrics(val_cat_data_age2plus[target_col], val_cat_data_age2plus['pred_prob'], threshold=0.5)

{'Sensitivity': np.float64(0.8688524590163934),
 'Specificity': np.float64(0.8977094446187637),
 'NPV': np.float64(0.9972115719762984),
 'ROC AUC': 0.9637821683375598,
 'PR AUC': 0.6441327429844189,
 'Precision': 0.13984168865435356,
 'TP': np.int64(53),
 'FN': np.int64(8),
 'TN': np.int64(2861),
 'FP': np.int64(326)}

#### Hyperparamter Tuning

In [46]:
param_space = {
    "depth": [4,5,6,7,8],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "l2_leaf_reg": [1,3,5,7,10]
}

import random
cat_age2plus_results = []

for i in range(20):   # only 20 trials needed
    params = {k: random.choice(v) for k,v in param_space.items()}

    model = CatBoostClassifier(
        iterations=2000,
        loss_function="Logloss",
        eval_metric="PRAUC",
        auto_class_weights="Balanced",
        early_stopping_rounds=100,
        random_seed=42,
        verbose=False,
        **params
    )

    model.fit(
        train_cat_data_age2plus[can_cols_age2plus],
        train_cat_data_age2plus[target_col],
        eval_set=(val_cat_data_age2plus[can_cols_age2plus], val_cat_data_age2plus[target_col]),
        cat_features= cat_features_age2plus
    )

    prob = model.predict_proba(val_cat_data_age2plus[can_cols_age2plus])[:,1]

    m = pecarn_metrics(val_cat_data_age2plus[target_col], prob, threshold=0.5)

    cat_age2plus_results.append({**params, **m})

In [47]:
pd.DataFrame(cat_age2plus_results)\
    .sort_values(["Sensitivity","PR AUC"], ascending=False)\
    .head()

,depth,learning_rate,l2_leaf_reg,Sensitivity,Specificity,NPV,ROC AUC,PR AUC,Precision,TP,FN,TN,FP
17,6,0.01,10,0.901639,0.848447,0.997786,0.955840,0.607662,0.102230,55,6,2704,483
7,5,0.01,7,0.901639,0.848447,0.997786,0.957481,0.603628,0.102230,55,6,2704,483
10,5,0.05,5,0.885246,0.899592,0.997564,0.965511,0.655981,0.144385,54,7,2867,320
6,7,0.03,3,0.885246,0.904612,0.997578,0.964960,0.651848,0.150838,54,7,2883,304
18,8,0.03,7,0.868852,0.925949,0.997296,0.963803,0.664249,0.183391,53,8,2951,236


Our best parameter combination is 
+ "depth" = 4,
+ "learning_rate" = 0.01
+ "l2_leaf_reg" = 7

#### Best

In [48]:
best_cat_model_age2plus = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=2000,
    learning_rate=0.01,
    depth=4,
    l2_leaf_reg=7,
    border_count=254,
    random_seed=42,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    verbose=200
)

best_cat_model_age2plus.fit(
    train_cat_data_age2plus[can_cols_age2plus],
    train_cat_data_age2plus[target_col],
    eval_set=(val_cat_data_age2plus[can_cols_age2plus], val_cat_data_age2plus[target_col]),
    cat_features= cat_features_age2plus
)

0:	test: 0.9196917	best: 0.9196917 (0)	total: 54.6ms	remaining: 1m 49s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9585302998
bestIteration = 27

Shrink model to first 28 iterations.


CatBoostClassifier(auto_class_weights='Balanced', border_count=254, depth=4, early_stopping_rounds=100, eval_metric='AUC', iterations=2000, l2_leaf_reg=7, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=200)

In [49]:
# find the best threshold on val set
val_cat_data_age2plus['pred_prob'] = best_cat_model_age2plus.predict_proba(val_cat_data_age2plus[can_cols_age2plus])[:, 1]
threshold_results = scan_thresholds(val_cat_data_age2plus[target_col], val_cat_data_age2plus['pred_prob'], thresholds=np.linspace(0.001, 0.2, 200))
threshold_results.sort_values(["youdenJ"], ascending=False).head(10)

,thr,sensitivity,specificity,youdenJ
0,0.001,1.0,0.0,0.0
137,0.138,1.0,0.0,0.0
127,0.128,1.0,0.0,0.0
128,0.129,1.0,0.0,0.0
129,0.130,1.0,0.0,0.0
130,0.131,1.0,0.0,0.0
131,0.132,1.0,0.0,0.0
132,0.133,1.0,0.0,0.0
133,0.134,1.0,0.0,0.0
134,0.135,1.0,0.0,0.0


The best threshold is at **0.2**

In [50]:
# train
train_cat_data_age2plus['pred_prob'] = best_cat_model_age2plus.predict_proba(train_cat_data_age2plus[can_cols_age2plus])[:, 1]
pecarn_metrics(train_cat_data_age2plus[target_col], train_cat_data_age2plus['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.0),
 'NPV': nan,
 'ROC AUC': 0.9411486338867804,
 'PR AUC': 0.48739595399875246,
 'Precision': 0.0187824403976423,
 'TP': np.int64(427),
 'FN': np.int64(0),
 'TN': np.int64(0),
 'FP': np.int64(22307)}

In [51]:
# val
val_cat_data_age2plus['pred_prob'] = best_cat_model_age2plus.predict_proba(val_cat_data_age2plus[can_cols_age2plus])[:, 1]
pecarn_metrics(val_cat_data_age2plus[target_col], val_cat_data_age2plus['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.0),
 'NPV': nan,
 'ROC AUC': 0.9585302998348826,
 'PR AUC': 0.5731010281380275,
 'Precision': 0.018780788177339903,
 'TP': np.int64(61),
 'FN': np.int64(0),
 'TN': np.int64(0),
 'FP': np.int64(3187)}

In [52]:
# test
test_cat_data_age2plus['pred_prob'] = best_cat_model_age2plus.predict_proba(test_cat_data_age2plus[can_cols_age2plus])[:, 1]
pecarn_metrics(test_cat_data_age2plus[target_col], test_cat_data_age2plus['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.0),
 'NPV': nan,
 'ROC AUC': 0.9439089127449115,
 'PR AUC': 0.4881281458612388,
 'Precision': 0.018780788177339903,
 'TP': np.int64(122),
 'FN': np.int64(0),
 'TN': np.int64(0),
 'FP': np.int64(6374)}

### Interpretability

In [53]:
# get feature importance
feature_importance = best_cat_model_age2plus.get_feature_importance(prettified=True)
feature_importance.head(20)

,Feature Id,Importances
0,AMSRepeat,38.547560
1,AMSOth,20.846453
2,LOCSeparate,19.090451
3,ActNorm,5.867461
4,GCSVerbal,2.730420
5,SFxBasHem,2.641424
6,GCSTotal,1.979148
7,SFxBasOto,1.144102
8,AMSSlow,0.960186
9,VomitLast,0.842369


**Remark:**

Compared with the CDR for group aged at two and older, we still have altered mental status, severe injury mechanism, severe headache and loss of consciousness among the most important features.

But, we don't see vomit and basilar skull fracture signs in the top20 features, which is surprising. 

Meanwhile, dizzy and trauma above the clavicles appear in the important features, both of which CDR don't include in both age group rule.

## Stability

In this section, I simply use the same strategy to test whether my feature selection and model selection strategy is stable with respect to GCS imputation strategy.

In the stricter dataset, I will not impelement imputing GCS strategy and drop the row with missing GCS component.


In [54]:
train_cat_data_new, val_cat_data_new, test_cat_data_new = preprocess_data(
    df=tbi_cleaned_new,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13= False, # include GCS 3-13
    if_fill_missing_gcs=False,
    gcs_fill_strategy="leave", # leave missing 
    if_one_hot_encode=False,
    drop_first_cat_in_ohe=False,
    if_drop_na_rows=False,
    if_handle_parent_child_missing=False,
    if_handle_missing_for_categories=False,
    missing_category_label="missing",
    if_add_flag_missing_cols=False,
    parent_missing_strategy="leave",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="leave",
    multi_missing_strategy="missing_category"
)

In [55]:
train_cat_data_new = train_cat_data_new.dropna(subset=['GCSVerbal', 'GCSMotor', 'GCSEye'])
val_cat_data_new = val_cat_data_new.dropna(subset=['GCSVerbal', 'GCSMotor', 'GCSEye'])
test_cat_data_new = test_cat_data_new.dropna(subset=['GCSVerbal', 'GCSMotor', 'GCSEye'])

In [56]:
# convert column types
train_cat_data_new[model_numeric_cols] = train_cat_data_new[model_numeric_cols].apply(pd.to_numeric, errors='coerce')
# convert category columns to string
train_cat_data_new[model_categorical_cols] = (
    train_cat_data_new[model_categorical_cols]
        .apply(lambda col: col.astype("string").astype("category"))
)
# fill NaN with `missing`
train_cat_data_new[model_categorical_cols] = (
    train_cat_data_new[model_categorical_cols]
        .apply(lambda col: col.cat.add_categories("missing").fillna("missing"))
)

# val
val_cat_data_new[model_numeric_cols] = val_cat_data_new[model_numeric_cols].apply(pd.to_numeric, errors='coerce')
val_cat_data_new[model_categorical_cols] = val_cat_data_new[model_categorical_cols].apply(lambda col: col.astype("string").astype("category"))
val_cat_data_new[model_categorical_cols] = (
    val_cat_data_new[model_categorical_cols]
        .apply(lambda col: col.cat.add_categories("missing").fillna("missing"))
)
# test
test_cat_data_new[model_numeric_cols] = test_cat_data_new[model_numeric_cols].apply(pd.to_numeric, errors='coerce')
test_cat_data_new[model_categorical_cols] = test_cat_data_new[model_categorical_cols].apply(lambda col: col.astype("string").astype("category"))
test_cat_data_new[model_categorical_cols] = (
    test_cat_data_new[model_categorical_cols]
        .apply(lambda col: col.cat.add_categories("missing").fillna("missing"))
)

In [57]:
# age group split
train_cat_data_age2plus_new = train_cat_data_new[train_cat_data_new['AgeTwoPlus'] == '2']
train_cat_data_age2minus_new = train_cat_data_new[train_cat_data_new['AgeTwoPlus'] == '1']

val_cat_data_age2plus_new = val_cat_data_new[val_cat_data_new['AgeTwoPlus'] == '2']
val_cat_data_age2minus_new = val_cat_data_new[val_cat_data_new['AgeTwoPlus'] == '1']

test_cat_data_age2plus_new = test_cat_data_new[test_cat_data_new['AgeTwoPlus'] == '2']
test_cat_data_age2minus_new = test_cat_data_new[test_cat_data_new['AgeTwoPlus'] == '1']

In [58]:
# drop age group indicator
train_cat_data_age2plus_new.drop(columns=['AgeTwoPlus'], inplace=True)
train_cat_data_age2minus_new.drop(columns=['AgeTwoPlus'], inplace=True)
val_cat_data_age2plus_new.drop(columns=['AgeTwoPlus'], inplace=True)
val_cat_data_age2minus_new.drop(columns=['AgeTwoPlus'], inplace=True)
test_cat_data_age2plus_new.drop(columns=['AgeTwoPlus'], inplace=True)
test_cat_data_age2minus_new.drop(columns=['AgeTwoPlus'], inplace=True) 

In [59]:
train_cat_data_new.shape, val_cat_data_new.shape, test_cat_data_new.shape

((29408, 126), (4193, 126), (8437, 126))

In [60]:
train_cat_data_age2plus_new.shape, train_cat_data_age2minus_new.shape, val_cat_data_age2plus_new.shape, val_cat_data_age2minus_new.shape, test_cat_data_age2plus_new.shape, test_cat_data_age2minus_new.shape

((22061, 125), (7347, 125), (3152, 125), (1041, 125), (6326, 125), (2111, 125))

### Age <2

In [61]:
best_cat_model_age2minus_new = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="PRAUC",
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    random_seed=42,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    verbose=200
)

best_cat_model_age2minus_new.fit(
    train_cat_data_age2minus_new[can_cols_age2minus],
    train_cat_data_age2minus_new[target_col],
    eval_set=(val_cat_data_age2minus_new[can_cols_age2minus], val_cat_data_age2minus_new[target_col]),
    cat_features= cat_features_age2minus
)

0:	learn: 0.8964449	test: 0.8182731	best: 0.8182731 (0)	total: 27.1ms	remaining: 54.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9287248048
bestIteration = 46

Shrink model to first 47 iterations.


CatBoostClassifier(auto_class_weights='Balanced', depth=8, early_stopping_rounds=100, eval_metric='PRAUC', iterations=2000, l2_leaf_reg=3, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=200)

In [62]:
# train
train_cat_data_age2minus_new['pred_prob'] = best_cat_model_age2minus_new.predict_proba(train_cat_data_age2minus_new[can_cols_age2minus])[:, 1]
pecarn_metrics(train_cat_data_age2minus_new[target_col], train_cat_data_age2minus_new['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(1.0),
 'Specificity': np.float64(0.6903644395361679),
 'NPV': np.float64(1.0),
 'ROC AUC': 0.967648485790718,
 'PR AUC': 0.5409320981730944,
 'Precision': 0.04390451832907076,
 'TP': np.int64(103),
 'FN': np.int64(0),
 'TN': np.int64(5001),
 'FP': np.int64(2243)}

In [63]:
# val
val_cat_data_age2minus_new['pred_prob'] = best_cat_model_age2minus_new.predict_proba(val_cat_data_age2minus_new[can_cols_age2minus])[:, 1]
pecarn_metrics(val_cat_data_age2minus_new[target_col], val_cat_data_age2minus_new['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.9230769230769231),
 'Specificity': np.float64(0.7208171206225681),
 'NPV': np.float64(0.9986522911051213),
 'ROC AUC': 0.943766836276564,
 'PR AUC': 0.3672384437674824,
 'Precision': 0.04013377926421405,
 'TP': np.int64(12),
 'FN': np.int64(1),
 'TN': np.int64(741),
 'FP': np.int64(287)}

In [64]:
# test
test_cat_data_age2minus_new['pred_prob'] = best_cat_model_age2minus_new.predict_proba(test_cat_data_age2minus_new[can_cols_age2minus])[:, 1]
pecarn_metrics(test_cat_data_age2minus_new[target_col], test_cat_data_age2minus_new['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.8),
 'Specificity': np.float64(0.6948582412301778),
 'NPV': np.float64(0.9958677685950413),
 'ROC AUC': 0.850232260131347,
 'PR AUC': 0.4245946262585704,
 'Precision': 0.036418816388467376,
 'TP': np.int64(24),
 'FN': np.int64(6),
 'TN': np.int64(1446),
 'FP': np.int64(635)}

### Age >= 2

In [65]:
best_cat_model_age2plus_new = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=2000,
    learning_rate=0.01,
    depth=4,
    l2_leaf_reg=7,
    border_count=254,
    random_seed=42,
    auto_class_weights="Balanced",
    early_stopping_rounds=100,
    verbose=200
)

best_cat_model_age2plus_new.fit(
    train_cat_data_age2plus_new[can_cols_age2plus],
    train_cat_data_age2plus_new[target_col],
    eval_set=(val_cat_data_age2plus_new[can_cols_age2plus], val_cat_data_age2plus_new[target_col]),
    cat_features= cat_features_age2plus
)

0:	test: 0.9106912	best: 0.9106912 (0)	total: 42.6ms	remaining: 1m 25s
200:	test: 0.9521848	best: 0.9522252 (196)	total: 5.79s	remaining: 51.8s
400:	test: 0.9570990	best: 0.9570990 (400)	total: 11.3s	remaining: 45.3s
600:	test: 0.9586102	best: 0.9586217 (599)	total: 16.7s	remaining: 38.8s
800:	test: 0.9592158	best: 0.9592389 (799)	total: 22.3s	remaining: 33.4s
1000:	test: 0.9600925	best: 0.9600983 (997)	total: 28s	remaining: 27.9s
1200:	test: 0.9604905	best: 0.9605828 (1138)	total: 34.2s	remaining: 22.7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9605827796
bestIteration = 1138

Shrink model to first 1139 iterations.


CatBoostClassifier(auto_class_weights='Balanced', border_count=254, depth=4, early_stopping_rounds=100, eval_metric='AUC', iterations=2000, l2_leaf_reg=7, learning_rate=0.01, loss_function='Logloss', random_seed=42, verbose=200)

In [66]:
# train
train_cat_data_age2plus_new['pred_prob'] = best_cat_model_age2plus_new.predict_proba(train_cat_data_age2plus_new[can_cols_age2plus])[:, 1]
pecarn_metrics(train_cat_data_age2plus_new[target_col], train_cat_data_age2plus_new['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.9800498753117207),
 'Specificity': np.float64(0.7709141274238227),
 'NPV': np.float64(0.9995211301328863),
 'ROC AUC': 0.9671890794712205,
 'PR AUC': 0.5809206097123875,
 'Precision': 0.07338935574229692,
 'TP': np.int64(393),
 'FN': np.int64(8),
 'TN': np.int64(16698),
 'FP': np.int64(4962)}

In [67]:
# val
val_cat_data_age2plus_new['pred_prob'] = best_cat_model_age2plus_new.predict_proba(val_cat_data_age2plus_new[can_cols_age2plus])[:, 1]
pecarn_metrics(val_cat_data_age2plus_new[target_col], val_cat_data_age2plus_new['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.9642857142857143),
 'Specificity': np.float64(0.7729328165374677),
 'NPV': np.float64(0.9991649269311065),
 'ROC AUC': 0.9605827796234773,
 'PR AUC': 0.6020950735719173,
 'Precision': 0.071334214002642,
 'TP': np.int64(54),
 'FN': np.int64(2),
 'TN': np.int64(2393),
 'FP': np.int64(703)}

In [68]:
# test
test_cat_data_age2plus_new['pred_prob'] = best_cat_model_age2plus_new.predict_proba(test_cat_data_age2plus_new[can_cols_age2plus])[:, 1]
pecarn_metrics(test_cat_data_age2plus_new[target_col], test_cat_data_age2plus_new['pred_prob'], threshold=0.2)

{'Sensitivity': np.float64(0.9824561403508771),
 'Specificity': np.float64(0.7755956213779781),
 'NPV': np.float64(0.9995850622406639),
 'ROC AUC': 0.9631273934998474,
 'PR AUC': 0.5524210063706602,
 'Precision': 0.07436918990703852,
 'TP': np.int64(112),
 'FN': np.int64(2),
 'TN': np.int64(4818),
 'FP': np.int64(1394)}

**Conclusion:**

In the case of catboost, my feature selection and threshold selection are insensitive to the imputation of GCS.

The model's performance is almost same in both two age groups.